# 신용카드 사기 탐지 딥러닝 모델
- 데이터셋: creditcard.csv
- 목표: 정상 거래(0)와 사기 거래(1) 이진 분류
- 프레임워크: TensorFlow/Keras

## 1. 라이브러리 불러오기

In [ ]:
# 데이터 처리
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# 사이킷런 전처리 및 평가
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

# 텐서플로우 / 케라스
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# 한글 폰트 설정 (macOS)
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 재현성을 위한 랜덤 시드 고정
np.random.seed(42)
tf.random.set_seed(42)

print('라이브러리 불러오기 완료')
print(f'TensorFlow 버전: {tf.__version__}')

## 2. 데이터 불러오기 & 탐색적 데이터 분석 (EDA)

In [ ]:
# CSV 파일 로드
df = pd.read_csv('creditcard.csv')

# 데이터 크기 확인 (행 수, 열 수)
print('=== 데이터 크기 ===')
print(f'행(거래 건수): {df.shape[0]:,}건')
print(f'열(피처 수):   {df.shape[1]}개')
print()

In [ ]:
# 컬럼별 데이터 타입 및 결측치 확인
print('=== 데이터 기본 정보 ===')
df.info()

In [ ]:
# 기술 통계 (평균, 표준편차, 최솟값, 최댓값 등)
print('=== 기술 통계 ===')
df.describe()

In [ ]:
# 결측치 확인
print('=== 결측치 확인 ===')
missing = df.isnull().sum()
print(f'결측치 있는 컬럼 수: {(missing > 0).sum()}개')
print('(모든 컬럼 결측치 없음)' if missing.sum() == 0 else missing[missing > 0])

In [ ]:
# Class 분포 확인 (정상 vs 사기)
print('=== Class 분포 ===')
class_counts = df['Class'].value_counts()
class_ratio  = df['Class'].value_counts(normalize=True) * 100

print(f'정상 거래 (Class=0): {class_counts[0]:,}건  ({class_ratio[0]:.4f}%)')
print(f'사기 거래 (Class=1): {class_counts[1]:,}건  ({class_ratio[1]:.4f}%)')
print(f'불균형 비율: {class_counts[0]/class_counts[1]:.0f}:1')

# 클래스 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 막대 그래프
axes[0].bar(['정상 (0)', '사기 (1)'], class_counts.values,
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class 분포 (거래 건수)', fontsize=13)
axes[0].set_ylabel('건수')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=11)

# 파이 차트
axes[1].pie(class_counts.values,
            labels=[f'정상\n{class_ratio[0]:.2f}%', f'사기\n{class_ratio[1]:.2f}%'],
            colors=['steelblue', 'tomato'],
            autopct='%1.2f%%', startangle=90)
axes[1].set_title('Class 비율', fontsize=13)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('class_distribution.png 저장 완료')

## 3. 데이터 전처리

In [ ]:
# Time, Amount만 StandardScaler로 정규화 (V1~V28은 PCA로 이미 정규화됨)
scaler = StandardScaler()
df[['Time', 'Amount']] = scaler.fit_transform(df[['Time', 'Amount']])

print('Time, Amount 정규화 완료')
print(df[['Time', 'Amount']].describe().round(4))

In [ ]:
# X(피처), y(타겟) 분리
X = df.drop(columns=['Class'])   # 30개 피처
y = df['Class']                  # 타겟 (0=정상, 1=사기)

print(f'X shape: {X.shape}  (샘플 수, 피처 수)')
print(f'y shape: {y.shape}')

In [ ]:
# 학습/테스트 데이터 분리 (stratify=y: 클래스 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'학습 데이터: {X_train.shape[0]:,}건')
print(f'테스트 데이터: {X_test.shape[0]:,}건')
print(f'학습 데이터 사기 비율: {y_train.mean()*100:.4f}%')
print(f'테스트 데이터 사기 비율: {y_test.mean()*100:.4f}%')

In [ ]:
# 클래스 불균형 처리: class_weight 계산
# 사기 클래스(1)에 더 높은 가중치를 부여해 소수 클래스를 더 잘 학습하도록 함
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))

print('=== 클래스 가중치 ===')
print(f'정상(0) 가중치: {class_weight[0]:.4f}')
print(f'사기(1) 가중치: {class_weight[1]:.4f}')
print(f'\n사기 클래스가 정상 클래스보다 약 {class_weight[1]/class_weight[0]:.0f}배 높은 가중치 적용')

## 4. 딥러닝 모델 구성

In [ ]:
# Sequential 모델 구성
model = Sequential([
    # 입력층: 30개 피처 입력
    Dense(64, input_dim=30, activation='relu', name='hidden_1'),
    Dropout(0.3, name='dropout_1'),           # 과적합 방지: 학습 시 30% 뉴런 무작위 비활성화

    # 은닉층 2
    Dense(32, activation='relu', name='hidden_2'),
    Dropout(0.3, name='dropout_2'),

    # 은닉층 3
    Dense(16, activation='relu', name='hidden_3'),

    # 출력층: 시그모이드 → 0~1 사이 확률 출력 (이진 분류)
    Dense(1, activation='sigmoid', name='output')
])

# 모델 컴파일
model.compile(
    optimizer='adam',                  # Adam 최적화 알고리즘
    loss='binary_crossentropy',        # 이진 분류 손실 함수
    metrics=['accuracy']               # 평가 지표: 정확도
)

# 모델 구조 요약 출력
print('=== 모델 구조 ===')
model.summary()

## 5. 모델 학습

In [ ]:
print('모델 학습 시작...')

history = model.fit(
    X_train, y_train,
    epochs=20,                    # 전체 데이터를 20번 반복 학습
    batch_size=512,               # 한 번에 512개 샘플씩 학습
    validation_split=0.2,         # 학습 데이터의 20%를 검증에 사용
    class_weight=class_weight,    # 불균형 처리: 사기 클래스에 높은 가중치 부여
    verbose=1
)

print('\n모델 학습 완료!')

## 6. 결과 시각화

In [ ]:
# Loss 그래프 (훈련 vs 검증)
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'],     label='훈련 Loss',  color='steelblue', linewidth=2)
plt.plot(history.history['val_loss'], label='검증 Loss',  color='tomato',    linewidth=2, linestyle='--')
plt.title('학습/검증 Loss 변화', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('loss_graph.png 저장 완료')

In [ ]:
# Accuracy 그래프 (훈련 vs 검증)
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'],     label='훈련 Accuracy', color='steelblue', linewidth=2)
plt.plot(history.history['val_accuracy'], label='검증 Accuracy', color='tomato',    linewidth=2, linestyle='--')
plt.title('학습/검증 Accuracy 변화', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.9, 1.01])
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('accuracy_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('accuracy_graph.png 저장 완료')

In [ ]:
# 테스트 데이터로 예측 수행
# 시그모이드 출력값 0.5 이상이면 사기(1), 미만이면 정상(0)으로 분류
y_pred_prob = model.predict(X_test, verbose=0)   # 확률값 (0~1)
y_pred      = (y_pred_prob >= 0.5).astype(int).flatten()  # 클래스 예측값

# 혼동 행렬 계산
cm = confusion_matrix(y_test, y_pred)

# 혼동 행렬 히트맵 시각화
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=['정상(0)', '사기(1)'],
    yticklabels=['정상(0)', '사기(1)'],
    linewidths=0.5, linecolor='gray'
)
plt.title('혼동 행렬 (Confusion Matrix)', fontsize=14)
plt.xlabel('예측값', fontsize=12)
plt.ylabel('실제값', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('confusion_matrix.png 저장 완료')

# 혼동 행렬 수치 해석
tn, fp, fn, tp = cm.ravel()
print(f'\n정확히 정상으로 분류 (TN): {tn:,}건')
print(f'사기를 정상으로 오분류 (FN): {fn}건  ← 놓친 사기!')
print(f'정상을 사기로 오분류 (FP): {fp}건')
print(f'정확히 사기로 분류 (TP): {tp}건')

In [ ]:
# Classification Report 출력 (Precision, Recall, F1-score)
print('=== Classification Report ===')
print(classification_report(
    y_test, y_pred,
    target_names=['정상 (0)', '사기 (1)']
))

# 테스트 Loss / Accuracy
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'테스트 Loss:     {test_loss:.4f}')
print(f'테스트 Accuracy: {test_acc*100:.2f}%')

## 7. 새로운 거래 데이터 예측 (일반화)

In [ ]:
# 새로운 거래 데이터 2건 생성
# - 거래1: 일반적인 정상 거래 패턴 (V 값이 작고 Amount 보통)
# - 거래2: 이상 거래 패턴 (V1, V3, V10 등 사기에서 자주 나타나는 음수 값 강조)

# 학습에 사용한 스케일러로 Time, Amount 정규화
# 여기서는 이미 정규화된 데이터프레임 기준값을 직접 사용

new_transactions = pd.DataFrame({
    'Time':   [ 0.0,   0.5],      # 정규화된 시간 값
    'V1':     [ 1.2,  -3.8],      # 사기일 때 V1은 음수 경향
    'V2':     [ 0.3,   2.1],
    'V3':     [ 0.8,  -4.5],      # 사기일 때 V3은 강한 음수 경향
    'V4':     [ 0.2,   3.9],
    'V5':     [-0.1,  -1.2],
    'V6':     [ 0.4,  -1.5],
    'V7':     [ 0.1,  -5.0],      # 사기 거래에서 자주 나타나는 이상값
    'V8':     [ 0.0,   0.9],
    'V9':     [-0.2,  -2.0],
    'V10':    [ 0.3,  -5.5],      # 사기 판별에 강한 영향을 미치는 피처
    'V11':    [ 0.5,   2.7],
    'V12':    [ 0.1,  -5.2],
    'V13':    [ 0.2,   0.1],
    'V14':    [-0.1,  -7.0],      # V14는 사기 탐지 중요도가 높은 피처
    'V15':    [ 0.3,   0.2],
    'V16':    [ 0.0,  -3.1],
    'V17':    [-0.2,  -5.8],
    'V18':    [ 0.1,  -1.0],
    'V19':    [ 0.2,   0.5],
    'V20':    [ 0.0,   0.1],
    'V21':    [-0.1,   0.8],
    'V22':    [ 0.1,   0.3],
    'V23':    [ 0.0,  -0.2],
    'V24':    [ 0.2,   0.1],
    'V25':    [ 0.1,  -0.1],
    'V26':    [ 0.0,   0.2],
    'V27':    [ 0.0,   0.1],
    'V28':    [ 0.0,   0.0],
    'Amount': [ 0.1,   2.5],      # 정규화된 거래 금액
})

# 열 순서를 학습 데이터와 동일하게 맞춤
new_transactions = new_transactions[X.columns]

# 예측 수행
pred_prob  = model.predict(new_transactions, verbose=0).flatten()
pred_class = (pred_prob >= 0.5).astype(int)

# 결과 출력
print('=== 새로운 거래 예측 결과 ===')
for i in range(len(new_transactions)):
    label = '사기 거래 의심!' if pred_class[i] == 1 else '정상 거래'
    print(f'거래 {i+1}: 사기 확률 = {pred_prob[i]*100:.2f}%  →  [{label}]')